# Lean Server demo

Build and start the server

| Endpoint | Purpose |
| --- | --- |
| `GET /healthz` | HTTP liveness |
| `GET /readyz` | Worker readiness |
| `POST /check` | Compile Lean code |
| `POST /verify_proof` | Verify a candidate against a statement |



In [4]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

# For local use, set LEAN_SERVER_URL=http://127.0.0.1:8000.
BASE_URL = os.environ.get("LEAN_SERVER_URL", "http://159.226.47.219:9123")

# A 120-second budget uses normal workers under default routing.
VERIFY_TIMEOUT_SECONDS = 120


class APIError(RuntimeError):
    def __init__(self, status, body):
        self.status = status
        self.body = body
        super().__init__(f"HTTP {status}: {body}")


def api(path, payload=None):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    # Allow more time than the total queue-and-execution budget.
    default_budget = 600 if path == "/verify_proof" else 30
    timeout = 10 if payload is None else payload.get("timeout_seconds", default_budget) + 15
    paths = [path]
    # Older deployments expose these operations under /api/v1.
    # Only a missing route triggers fallback; other errors are returned directly.
    if payload is not None and path in ("/check", "/verify_proof"):
        paths.append("/api/v1" + path)
    for index, request_path in enumerate(paths):
        request = Request(
            BASE_URL.rstrip("/") + request_path,
            data=data,
            method="GET" if payload is None else "POST",
            headers={"Content-Type": "application/json"},
        )
        try:
            with urlopen(request, timeout=timeout) as response:
                return json.load(response)
        except HTTPError as exc:
            body = exc.read().decode("utf-8")
            exc.close()
            if exc.code == 404 and index + 1 < len(paths):
                continue
            try:
                body = json.loads(body)
            except json.JSONDecodeError:
                pass
            raise APIError(exc.code, body) from exc
        except URLError as exc:
            raise RuntimeError(f"Cannot connect to {BASE_URL}; check the URL and server.") from exc


def show(result):
    print(json.dumps(result, indent=2, ensure_ascii=False))


print(f"Using Lean Server: {BASE_URL}")


Using Lean Server: http://159.226.47.219:9123


## 1. Liveness and readiness

`/healthz` reports service state. 
`/readyz` returns 200 when the pool is running
with at least one ready worker, otherwise 503. 


In [5]:
show(api("/healthz"))
show(api("/readyz"))


{
  "status": "ok",
  "lean_version": "4.30.0",
  "pool": {
    "state": "running",
    "worker_count": 32,
    "ready_workers": 32,
    "active_workers": 0,
    "queue_depth": 0,
    "queue_capacity": 64,
    "replacements": 0
  }
}
{
  "status": "ready",
  "pool": {
    "state": "running",
    "worker_count": 32,
    "ready_workers": 32,
    "active_workers": 0,
    "queue_depth": 0,
    "queue_capacity": 64,
    "replacements": 0
  }
}


## 2. Compile a valid proof

This example shows the correct code can pass lean verification.


In [6]:
result = api("/check", {
    "code": "import Mathlib\nexample : 1 + 1 = 2 := by norm_num",
    "timeout_seconds": 30,
})
show(result)
assert result["okay"], result


{
  "okay": true,
  "timed_out": false,
  "time_ms": 17.426028847694397,
  "lean_version": "4.30.0",
  "warnings": [],
  "errors": [],
  "timings": {
    "total_ms": 17.426028847694397,
    "queue_ms": 1.1579078476943963,
    "compile_ms": 16.268121
  }
}


## 3. Inspect compile errors

This example shows the wrong code can not pass the verification.

In [7]:
result = api("/check", {"code": "example : Nat := True.intro"})
show(result)
assert not result["okay"] and result["errors"], result


{
  "okay": false,
  "timed_out": false,
  "time_ms": 19.20384168624878,
  "lean_version": "4.30.0",
  "warnings": [],
  "errors": [
    {
      "severity": "error",
      "message": "Type mismatch\n  True.intro\nhas type\n  True\nof sort `Prop` but is expected to have type\n  ℕ\nof sort `Type`",
      "file_name": "<stdin>",
      "start": {
        "line": 1,
        "column": 17
      },
      "end": {
        "line": 1,
        "column": 27
      }
    }
  ],
  "timings": {
    "total_ms": 19.20384168624878,
    "queue_ms": 1.3031406862487778,
    "compile_ms": 17.900701
  }
}


## 4. Control `sorry` in compilation

`allow_sorry` defaults to `false`. Set it to `true` to check incomplete code;
acceptance then does not mean the proof is complete. Strict verification cannot
permit `sorry`.


In [8]:
incomplete = "theorem answer : True := by sorry"
for allow_sorry in (False, True):
    print(f"allow_sorry={allow_sorry}")
    result = api("/check", {
        "code": incomplete,
        "allow_sorry": allow_sorry,
    })
    show(result)
    assert result["okay"] == allow_sorry, result


allow_sorry=False
{
  "okay": false,
  "timed_out": false,
  "time_ms": 9.320098906755447,
  "lean_version": "4.30.0",
  "warnings": [
    {
      "severity": "warning",
      "message": "declaration uses `sorry`",
      "file_name": "<stdin>",
      "start": {
        "line": 1,
        "column": 8
      },
      "end": {
        "line": 1,
        "column": 14
      }
    }
  ],
  "errors": [
    {
      "severity": "error",
      "message": "declaration uses `sorry`, but allow_sorry is false",
      "file_name": "<stdin>",
      "start": {
        "line": 1,
        "column": 8
      },
      "end": {
        "line": 1,
        "column": 14
      }
    }
  ],
  "timings": {
    "total_ms": 9.320098906755447,
    "queue_ms": 1.1164029067554466,
    "compile_ms": 8.203696
  }
}
allow_sorry=True
{
  "okay": true,
  "timed_out": false,
  "time_ms": 7.709041237831116,
  "lean_version": "4.30.0",
  "warnings": [
    {
      "severity": "warning",
      "message": "declaration uses `sorry`

## 5. Verify against a statement

`formal_statement` defines the targets, with `sorry` or complete proofs.
`content` supplies complete proofs. It can help check if `formal_statement` and `content` describe the same thing.

In [9]:
formal_statement = "theorem answer : True := by sorry"


def verify(content, *, statement=formal_statement, **options):
    payload = {
        "formal_statement": statement,
        "content": content,
        "environment": "lean-4.30.0",
        "timeout_seconds": VERIFY_TIMEOUT_SECONDS,
    }
    payload.update(options)
    return api("/verify_proof", payload)


try:
    result = verify("theorem answer : True := True.intro")
except APIError as exc:
    if exc.status == 404:
        raise RuntimeError(
            f"{BASE_URL} has no /verify_proof route. Update the server "
            "or point BASE_URL to the current local build. Verification did not run."
        ) from exc
    raise
show(result)
assert result.get("okay") is True, result


{
  "okay": true,
  "content": "theorem answer : True := True.intro",
  "lean_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "tool_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "timings": {
    "total_ms": 12.242693454027176,
    "formal_statement_ms": 6.254173,
    "declarations_ms": 0.214023,
    "candidate_ms": 4.655124
  },
  "failed_declarations": []
}


## 6. Reject incomplete proofs and changed targets

Code may compile without proving the specified target. The changed statement below
passes `/check` but fails verification.

In [10]:
changed_statement = "theorem answer : 1 = 1 := rfl"
compiled = api("/check", {"code": changed_statement})
show(compiled)
assert compiled["okay"], compiled

for label, candidate in (
    ("Incomplete proof", formal_statement),
    ("Changed target", changed_statement),
):
    print(label)
    result = verify(candidate)
    show(result)
    assert result.get("okay") is False, result


{
  "okay": true,
  "timed_out": false,
  "time_ms": 8.456073701381683,
  "lean_version": "4.30.0",
  "warnings": [],
  "errors": [],
  "timings": {
    "total_ms": 8.456073701381683,
    "queue_ms": 0.8963647013816836,
    "compile_ms": 7.559709
  }
}
Incomplete proof
{
  "okay": false,
  "content": "theorem answer : True := by sorry",
  "lean_messages": {
    "errors": [],
    "warnings": [
      "declaration uses `sorry`"
    ],
    "infos": []
  },
  "tool_messages": {
    "errors": [
      "Declaration 'answer' is incomplete (uses 'sorry')"
    ],
    "warnings": [],
    "infos": []
  },
  "timings": {
    "total_ms": 13.075314462184906,
    "formal_statement_ms": 6.561857,
    "declarations_ms": 0.165592,
    "candidate_ms": 5.263852
  },
  "failed_declarations": [
    "answer"
  ]
}
Changed target
{
  "okay": false,
  "content": "theorem answer : 1 = 1 := rfl",
  "lean_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "tool_messages": {
    "errors": [
  

## 7. Complete statements need no `sorry`

A complete formal proof may be replaced by another valid proof. Changed targets
and incomplete candidates are still rejected.


In [11]:
complete_statement = "theorem answer : True := True.intro"
for label, candidate, expected in (
    ("Different valid proof", "theorem answer : True := by constructor", True),
    ("Changed target", "theorem answer : 1 = 1 := rfl", False),
    ("Candidate with sorry", "theorem answer : True := by sorry", False),
):
    print(label)
    result = verify(candidate, statement=complete_statement)
    show(result)
    assert result.get("okay") is expected, result


Different valid proof
{
  "okay": true,
  "content": "theorem answer : True := by constructor",
  "lean_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "tool_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "timings": {
    "total_ms": 11.95066049695015,
    "formal_statement_ms": 6.715159,
    "declarations_ms": 0.185472,
    "candidate_ms": 3.931396
  },
  "failed_declarations": []
}
Changed target
{
  "okay": false,
  "content": "theorem answer : 1 = 1 := rfl",
  "lean_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "tool_messages": {
    "errors": [
      "Theorem 'answer' does not match expected signature"
    ],
    "warnings": [],
    "infos": []
  },
  "timings": {
    "total_ms": 12.416660785675049,
    "formal_statement_ms": 4.973078,
    "declarations_ms": 0.216972,
    "candidate_ms": 6.108312
  },
  "failed_declarations": [
    "answer"
  ]
}
Candidate with sorry
{
  "okay": false,
  "content": "t

## 8. Definitional and structural equality

`(n : Nat := 60)` uses `optParam Nat 60`, definitionally equal to `Nat`.
Default comparison accepts this candidate; `use_def_eq=False` rejects its signature.
Structural comparison normalizes universe parameters by position and compares
elaborated expressions, not source text. Definitional equality does not preserve
default-argument call syntax.


In [12]:
default_argument_statement = "theorem target (n : Nat := 60) : n = n := by sorry"
plain_argument_candidate = "theorem target (n : Nat) : n = n := rfl"
for label, options, expected in (
    ("Default definitional equality", {}, True),
    ("Structural equality", {"use_def_eq": False}, False),
    ("Default remains definitional equality", {}, True),
):
    print(label)
    result = verify(plain_argument_candidate, statement=default_argument_statement, **options)
    show(result)
    assert result.get("okay") is expected, result
    if not expected:
        assert "target" in result["failed_declarations"], result


Default definitional equality
{
  "okay": true,
  "content": "theorem target (n : Nat) : n = n := rfl",
  "lean_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "tool_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "timings": {
    "total_ms": 14.478180557489395,
    "formal_statement_ms": 8.52964,
    "declarations_ms": 0.403615,
    "candidate_ms": 4.29962
  },
  "failed_declarations": []
}
Structural equality
{
  "okay": false,
  "content": "theorem target (n : Nat) : n = n := rfl",
  "lean_messages": {
    "errors": [],
    "warnings": [],
    "infos": []
  },
  "tool_messages": {
    "errors": [
      "Theorem 'target' does not match expected signature"
    ],
    "warnings": [],
    "infos": []
  },
  "timings": {
    "total_ms": 14.47279378771782,
    "formal_statement_ms": 8.644882,
    "declarations_ms": 0.251263,
    "candidate_ms": 4.497773
  },
  "failed_declarations": [
    "target"
  ]
}
Default remains definitional equal